In [1]:
# Importing necessary libraries
import pandas as pd
import sqlite3

## Task 1: Data Gathering and Combination

In [2]:
# Establishing a connection to the SQLite database
conn = sqlite3.connect('level3_final_project_database.db')

In [3]:
# Exploring the members table
members_df = pd.read_sql_query("SELECT * FROM members", conn)
print("Members Table:")
print(members_df.head())

Members Table:
   member_id first_name last_name  grade neighborhood membership_status  \
0       1001      Salma   Ibrahim    8.0        Maadi            Active   
1       1002      Fares     Saleh    9.0        Maadi            Active   
2       1003     Bassel    Hegazy    6.0        Maadi            Active   
3       1004      Fares     Wahba    7.0        Maadi          inactive   
4       1005    Youssef     Halim    9.0        Maadi            Active   

    join_date  
0  2023-04-05  
1         NaN  
2  2025-04-23  
3  2024-10-09  
4  2024-05-05  


In [4]:
# Exploring checkouts table
checkouts_df = pd.read_sql_query("SELECT * FROM checkouts", conn)
print("\nCheckouts Table:")
print(checkouts_df.head())


Checkouts Table:
   checkout_id  member_id  book_id checkout_date return_date
0         9263       1047      517    2024-10-21  2024-11-07
1         9340       1072      513    2025-08-24  2025-09-01
2         9231       1053      523    2024-02-04  2024-02-16
3         9129       1032      513    2025-06-21  2025-06-29
4         9370       1079      511    2025-11-11  2025-12-03


In [5]:
# Exploring books table
books_df = pd.read_sql_query("SELECT * FROM books", conn)
print("\nBooks Table:")
print(books_df.head())


Books Table:
   book_id                title         author
0      501      The Silver Kite  Amina Darwish
1      502       Desert Compass  Amina Darwish
2      503    The Lantern Maker   Adel Roushdy
3      504  Rooftop Astronomers   Adel Roushdy
4      505  Letters to the Nile      Aya Hafez


In [6]:
# First question: How much is each member borrowing?
each_borrowing_count = pd.read_sql_query('''
SELECT
    members.member_id,
    members.first_name,
    members.last_name,
    COUNT(checkouts.checkout_id) AS borrowing_count
FROM members
LEFT JOIN checkouts ON members.member_id = checkouts.member_id
GROUP BY members.member_id, members.first_name, members.last_name
''', conn)
print("\nMember Checkouts Table:")
print(each_borrowing_count.head())
print(each_borrowing_count.shape)


Member Checkouts Table:
   member_id first_name last_name  borrowing_count
0       1001      Salma   Ibrahim                1
1       1002      Fares     Saleh                2
2       1003     Bassel    Hegazy                9
3       1004      Fares     Wahba                0
4       1005    Youssef     Halim                3
(80, 4)


In [7]:
# Second question: Which books match a chosen author pattern?
pattern_authors_df = pd.read_sql_query("SELECT * FROM books WHERE author LIKE 'A%'", conn)
print("\nBooks by Authors Starting with 'A':")
print(pattern_authors_df)


Books by Authors Starting with 'A':
   book_id                title         author
0      501      The Silver Kite  Amina Darwish
1      502       Desert Compass  Amina Darwish
2      503    The Lantern Maker   Adel Roushdy
3      504  Rooftop Astronomers   Adel Roushdy
4      505  Letters to the Nile      Aya Hafez
5      506  The Paper Boat Club      Aya Hafez


In [8]:
# Third question: What are the most five popular books?
popular_books_df = pd.read_sql_query('''
SELECT
    books.title,
    COUNT(checkouts.checkout_id) as checkout_count
FROM books
LEFT JOIN checkouts ON books.book_id = checkouts.book_id
GROUP BY books.title, books.book_id 
ORDER BY checkout_count DESC
LIMIT 5''', conn)
print("\nTop 5 Most Popular Books:")
print(popular_books_df)


Top 5 Most Popular Books:
                    title  checkout_count
0         The Silver Kite              57
1   Fossils and Fireflies              55
2  Circuits for Beginners              46
3        Kites Over Cairo              38
4    Storms and Sailboats              25


In [9]:
# Fourth question: Who are the most active readers?
active_readers_df = pd.read_sql_query('''
SELECT 
    members.first_name,
    members.last_name,
    members.member_id,
    COUNT(checkouts.checkout_id) as Borrowing_count
FROM members
JOIN checkouts ON members.member_id = checkouts.member_id
GROUP BY members.member_id ORDER BY Borrowing_count DESC 
LIMIT 10;
''', conn)
print("\nTop 10 Most Active Readers:")
print(active_readers_df)


Top 10 Most Active Readers:
  first_name last_name  member_id  Borrowing_count
0        Aya     Wahba       1034               25
1     Sherif     Saleh       1044               21
2       Ziad     Saleh       1008               19
3    Mostafa     Fouad       1027               18
4       Nour     Nabil       1010               18
5       Adam     Fahmy       1065               17
6    Youssef    Hegazy       1024               17
7      Ahmed    Shafik       1018               17
8       Sara    Rashad       1047               16
9       Reem     Osman       1030               16


In [10]:
# Fifth question: What does a neighborhood's activity look like further back in time?
neighborhood_activity_df = pd.read_sql_query("""
SELECT 
    checkouts.checkout_date,
    members.first_name,
    members.last_name,
    members.member_id
FROM checkouts
JOIN members ON checkouts.member_id = members.member_id
WHERE members.neighborhood = 'Maadi'
ORDER BY checkouts.checkout_date DESC
LIMIT 10
OFFSET 10;
""", conn)
print("\nNeighborhood Activity for 'Maadi':")
print(neighborhood_activity_df)


Neighborhood Activity for 'Maadi':
  checkout_date first_name last_name  member_id
0    2025-09-04     Bassel    Hegazy       1003
1    2025-08-25       Adam      Badr       1017
2    2025-08-23       Ziad     Saleh       1008
3    2025-08-21     Bassel    Hegazy       1003
4    2025-08-19      Ahmed    Shafik       1018
5    2025-08-08      Hamza     Sabry       1015
6    2025-08-04      Ahmed    Shafik       1018
7    2025-07-27       Ziad     Fouad       1013
8    2025-07-22     Bassel    Hegazy       1003
9    2025-07-22     Hassan     Saleh       1009


In [11]:
conn.close()

In [12]:
# Stage1: Merging members and checkouts table
stage1 = pd.merge(members_df, checkouts_df, on='member_id', how='left')
print("Stage1 DataFrame\n")
print(stage1.head())

Stage1 DataFrame

   member_id first_name last_name  grade neighborhood membership_status  \
0       1001      Salma   Ibrahim    8.0        Maadi            Active   
1       1002      Fares     Saleh    9.0        Maadi            Active   
2       1002      Fares     Saleh    9.0        Maadi            Active   
3       1003     Bassel    Hegazy    6.0        Maadi            Active   
4       1003     Bassel    Hegazy    6.0        Maadi            Active   

    join_date  checkout_id  book_id checkout_date return_date  
0  2023-04-05       9025.0    525.0    2024-10-24  2024-11-07  
1         NaN       9013.0    501.0    2024-02-16  2024-02-29  
2         NaN       9095.0    507.0    2024-06-25  2024-07-07  
3  2025-04-23       9051.0    513.0    2025-07-22  2025-08-18  
4  2025-04-23       9068.0    501.0    2025-11-14  2025-12-03  


In [13]:
# Book Details
book_details = pd.read_json('level3_final_project_book_catalog')
print("Book details:\n")
print(book_details.head())

Book details:

   book_id       genre  pages  publication_year            publisher
0      501   Adventure    128            2017.0           Nile Press
1      502   Adventure    109            2018.0          Delta House
2      503  Historical    259               NaN           Nile Press
3      504     Science    319            2009.0  Cairo Young Readers
4      505  Historical    216            2024.0          Oasis Books


In [14]:
# Stage2: Merge The Two DataFrames 
stage2 = pd.merge(stage1, book_details, on='book_id', how='left')
print("Stage2 DataFrame:\n")
print(stage2.head())

Stage2 DataFrame:

   member_id first_name last_name  grade neighborhood membership_status  \
0       1001      Salma   Ibrahim    8.0        Maadi            Active   
1       1002      Fares     Saleh    9.0        Maadi            Active   
2       1002      Fares     Saleh    9.0        Maadi            Active   
3       1003     Bassel    Hegazy    6.0        Maadi            Active   
4       1003     Bassel    Hegazy    6.0        Maadi            Active   

    join_date  checkout_id  book_id checkout_date return_date      genre  \
0  2023-04-05       9025.0    525.0    2024-10-24  2024-11-07  Adventure   
1         NaN       9013.0    501.0    2024-02-16  2024-02-29  Adventure   
2         NaN       9095.0    507.0    2024-06-25  2024-07-07    Science   
3  2025-04-23       9051.0    513.0    2025-07-22  2025-08-18    Science   
4  2025-04-23       9068.0    501.0    2025-11-14  2025-12-03  Adventure   

   pages  publication_year    publisher  
0  297.0            2015.0  Oas

In [15]:
# Scraping the HTML page to get event signup
scraped_df = pd.read_html('level3_final_project_event_signup')[0]
print("Scraped Event Signup DataFrame:\n")
print(scraped_df)

Scraped Event Signup DataFrame:

    Member ID  Book ID Checkout Date
0        1026      522    2025-07-11
1        1049      520    2025-07-11
2        1062      525    2025-07-05
3        1065      520    2025-07-07
4        1104      515    2025-07-07
5        1009      503    2025-07-09
6        1063      522    2025-07-07
7        1022      511    2025-07-12
8        1029      523    2025-07-09
9        1201      509    2025-07-10
10       1005      513    2025-07-10
11       1104      526    2025-07-05
12       1058      518    2025-07-08
13       1002      521    2025-07-08
14       1150      530    2025-07-10
15       1041      504    2025-07-05
16       1055      526    2025-07-11
17       1061      513    2025-07-06
18       1012      509    2025-07-11
19       1007      510    2025-07-07
20       1073      530    2025-07-07
21       1003      501    2025-07-08
22       1017      507    2025-07-11
23       1061      504    2025-07-06
24       1201      523    2025-07-08
25   

In [16]:
# Adjust the name of the columns of the scraped dataframe
print("Old Column Names:")
print(scraped_df.columns.tolist())

scraped_df.columns = [col.strip().lower().replace(' ', '_') for col in scraped_df.columns]

print("New Column Names:")
print(scraped_df.columns.tolist())


Old Column Names:
['Member ID', 'Book ID', 'Checkout Date']
New Column Names:
['member_id', 'book_id', 'checkout_date']


In [17]:
# Merge the scraped df to be ready to concat with the stage2 df
scraped_full = pd.merge(scraped_df, members_df, on='member_id', how='left')
scraped_full = pd.merge(scraped_full, book_details, on='book_id', how='left')

print(scraped_full.head())

   member_id  book_id checkout_date first_name last_name  grade neighborhood  \
0       1026      522    2025-07-11       Nada     Saleh    7.0    Nasr City   
1       1049      520    2025-07-11      Ahmed     Gamal    6.0   Heliopolis   
2       1062      525    2025-07-05      Tarek      Adel    7.0      Zamalek   
3       1065      520    2025-07-07       Adam     Fahmy    6.0      Zamalek   
4       1104      515    2025-07-07        NaN       NaN    NaN          NaN   

  membership_status   join_date            genre  pages  publication_year  \
0          inactive  2023-11-16          Mystery    104            2016.0   
1            Active  2023-06-10  Science Fiction    136            2009.0   
2            Active  2023-01-22        Adventure    297            2015.0   
3            Active  2025-07-12  Science Fiction    136            2009.0   
4               NaN         NaN           Poetry    316            2022.0   

             publisher  
0  Cairo Young Readers  
1     

In [18]:
# Concat the stage2 df and the scraped_full df

stage3 = pd.concat([stage2, scraped_full], ignore_index=True)
print("Stage3 DataFrame:\n")
print(stage3.head())
print('--' * 50)
print('The shape of the DataFrame(rows ,columns):',stage3.shape)
print('--' * 50)
print("Columns:\n",stage3.columns.tolist())

Stage3 DataFrame:

   member_id first_name last_name  grade neighborhood membership_status  \
0       1001      Salma   Ibrahim    8.0        Maadi            Active   
1       1002      Fares     Saleh    9.0        Maadi            Active   
2       1002      Fares     Saleh    9.0        Maadi            Active   
3       1003     Bassel    Hegazy    6.0        Maadi            Active   
4       1003     Bassel    Hegazy    6.0        Maadi            Active   

    join_date  checkout_id  book_id checkout_date return_date      genre  \
0  2023-04-05       9025.0    525.0    2024-10-24  2024-11-07  Adventure   
1         NaN       9013.0    501.0    2024-02-16  2024-02-29  Adventure   
2         NaN       9095.0    507.0    2024-06-25  2024-07-07    Science   
3  2025-04-23       9051.0    513.0    2025-07-22  2025-08-18    Science   
4  2025-04-23       9068.0    501.0    2025-11-14  2025-12-03  Adventure   

   pages  publication_year    publisher  
0  297.0            2015.0  Oas

In [19]:
# Exploring last five rows
print("Last five rows:\n")
print(stage3.tail())

Last five rows:

     member_id first_name last_name  grade neighborhood membership_status  \
430       1003     Bassel    Hegazy    6.0        Maadi            Active   
431       1017       Adam      Badr    9.0        Maadi            Active   
432       1061       Ziad     Fahmy    9.0      zamalek          Inactive   
433       1201        NaN       NaN    NaN          NaN               NaN   
434       1041       Nada    Rashad    7.0    Nasr City            Active   

      join_date  checkout_id  book_id checkout_date return_date       genre  \
430  2025-04-23          NaN    501.0    2025-07-08         NaN   Adventure   
431  2025-07-12          NaN    507.0    2025-07-11         NaN     Science   
432  2023-04-27          NaN    504.0    2025-07-06         NaN     Science   
433         NaN          NaN    523.0    2025-07-08         NaN  Historical   
434  2024-04-02          NaN    512.0    2025-07-06         NaN   Adventure   

     pages  publication_year            publi

In [20]:
stage3.to_csv('task1_combined_data.csv', index=False)
print("Data Saved Successfully.")

Data Saved Successfully.


## Task2: Data Integrity

In [21]:
# Load The combined dataset
full_df = pd.read_csv('task1_combined_data.csv')
print("Data loaded successfully.")

Data loaded successfully.


In [22]:
combined_df = full_df.copy()

### Inspect the data

In [23]:
# Exploring the first five rows
print(combined_df.head())

   member_id first_name last_name  grade neighborhood membership_status  \
0       1001      Salma   Ibrahim    8.0        Maadi            Active   
1       1002      Fares     Saleh    9.0        Maadi            Active   
2       1002      Fares     Saleh    9.0        Maadi            Active   
3       1003     Bassel    Hegazy    6.0        Maadi            Active   
4       1003     Bassel    Hegazy    6.0        Maadi            Active   

    join_date  checkout_id  book_id checkout_date return_date      genre  \
0  2023-04-05       9025.0    525.0    2024-10-24  2024-11-07  Adventure   
1         NaN       9013.0    501.0    2024-02-16  2024-02-29  Adventure   
2         NaN       9095.0    507.0    2024-06-25  2024-07-07    Science   
3  2025-04-23       9051.0    513.0    2025-07-22  2025-08-18    Science   
4  2025-04-23       9068.0    501.0    2025-11-14  2025-12-03  Adventure   

   pages  publication_year    publisher  
0  297.0            2015.0  Oasis Books  
1  128.0

In [24]:
# Exploring the missing values
print(combined_df.isnull().sum())
print(f"The total number of missing values: {combined_df.isnull().sum().sum()}")

member_id              0
first_name             5
last_name              5
grade                 43
neighborhood           5
membership_status      5
join_date             12
checkout_id           44
book_id               18
checkout_date         18
return_date          109
genre                 18
pages                 18
publication_year      53
publisher             18
dtype: int64
The total number of missing values: 371


In [25]:
# Exploring number of duplicated records
print(combined_df.duplicated().sum())

8


In [26]:
# Exploring inconsistent formatting
print(combined_df['membership_status'].unique())

<StringArray>
['Active', 'inactive', 'Inactive', 'active', 'INACTIVE', nan]
Length: 6, dtype: str


In [27]:
print(combined_df['first_name'].unique())

<StringArray>
[  'Salma',   'Fares',  'Bassel', 'Youssef',   'Layla',  'Sherif',    'Ziad',
  'Hassan',    'Nour',   'Menna',     'Ali',  'Habiba',   'Hamza',    'Dina',
    'Adam',   'Ahmed', 'Mostafa',    'Rana',    'Hana',    'Nada',    'Reem',
  'Marwan',     'Aya',   'Karim',    'Lina',   'Retaj',     'Mai',    'Amir',
    'Sara',  'Farida',   'Tarek',    'Jana',    'Seif',  'Yassin',   'Malak',
       nan]
Length: 36, dtype: str


In [28]:
print(combined_df['neighborhood'].unique())

<StringArray>
[     'Maadi',     'Maadi ',  'Nasr City', 'Nasr  City',  'NASR CITY',
 'HELIOPOLIS', 'Heliopolis',    'zamalek',    'Zamalek',     'Shubra',
          nan]
Length: 11, dtype: str


In [29]:
print(combined_df['genre'].unique())

<StringArray>
[      'Adventure',         'Science',      'Historical',      'Friendship',
         'Mystery',               nan,          'Nature', 'Science Fiction',
          'Poetry']
Length: 9, dtype: str


In [30]:
print(combined_df['publisher'].unique())

<StringArray>
['Oasis Books', 'Nile Press', 'Cairo Young Readers', nan, 'Delta House']
Length: 5, dtype: str


In [31]:
# Exploring orphan records
registrated_ids = set(members_df['member_id'])
all_ids = set(combined_df['member_id'].dropna())
orphan_ids = all_ids - registrated_ids
print(f"Number of orphan ids: {len(orphan_ids)}")
print(f"Orpahn ids found: {orphan_ids}")
orphan_rows_count = combined_df[combined_df['member_id'].isin(orphan_ids)].shape[0]
print(f"number of rows affected by orphan ids: {orphan_rows_count}")

Number of orphan ids: 3
Orpahn ids found: {1104, 1201, 1150}
number of rows affected by orphan ids: 5


### Handle The Inconsistent Formatting

In [32]:
# Fixing inconsistent formatting in membership_status column

print(combined_df['membership_status'].unique()) # Display the unique values in the column before cleaning

print("--" * 30)

combined_df['membership_status'] = combined_df['membership_status'].str.strip().str.title()

print(combined_df['membership_status'].unique()) # Display the unique values in the column after cleaning

<StringArray>
['Active', 'inactive', 'Inactive', 'active', 'INACTIVE', nan]
Length: 6, dtype: str
------------------------------------------------------------
<StringArray>
['Active', 'Inactive', nan]
Length: 3, dtype: str


In [33]:
# Fixing inconsistent formatting in neighborhood column

print(combined_df['neighborhood'].unique()) # display the unique values in the column before cleaning

print("--" * 30)

combined_df['neighborhood'] = combined_df['neighborhood'].str.strip().str.title().replace("Nasr  City", "Nasr City")

print(combined_df['neighborhood'].unique()) # display the unique values in the column after cleaning

<StringArray>
[     'Maadi',     'Maadi ',  'Nasr City', 'Nasr  City',  'NASR CITY',
 'HELIOPOLIS', 'Heliopolis',    'zamalek',    'Zamalek',     'Shubra',
          nan]
Length: 11, dtype: str
------------------------------------------------------------
<StringArray>
['Maadi', 'Nasr City', 'Heliopolis', 'Zamalek', 'Shubra', nan]
Length: 6, dtype: str


### Handle The Duplicates

In [34]:
print("The number of True duplicates before cleaning is:", combined_df.duplicated().sum()) # Display the number of duplicates before cleaning

# Remove the duplicates
combined_df = combined_df.drop_duplicates()

print("The number of True duplicates after cleaning is:", combined_df.duplicated().sum()) # Display the number of duplicates after cleaning

The number of True duplicates before cleaning is: 8
The number of True duplicates after cleaning is: 0


### Handle The Missing Values

In [35]:
# Filling the missing values
combined_df['first_name'] = combined_df['first_name'].fillna("Unregistered") # Fill the missing values in first_name column with 'unregistered' becaus the member didn't register his name
combined_df['last_name'] = combined_df['last_name'].fillna("Member") # Fill the missing values in last_name column with 'Member' to make the full name Unregistered Member
combined_df['neighborhood'] = combined_df['neighborhood'].fillna("Unknown") # Fill the missing values in neighborhood column with Unknown
combined_df['membership_status'] = combined_df['membership_status'].fillna('Unknown') # Fill the missing values in membershp_status column with Unknown
combined_df['join_date'] = combined_df['join_date'].fillna('Unknown') # Fill the missing values in join_date column with Unknown
combined_df['grade'] = combined_df['grade'].fillna("N/A") # Fill the missing values in grade column with N/A
combined_df['checkout_id'] = combined_df['checkout_id'].fillna('N/A') # Fill the missing values in checkout_id column with N/A
combined_df['book_id'] = combined_df['book_id'].fillna('Unknown') # Fill the missing values in book_id column with N/A
combined_df['checkout_date'] = combined_df['checkout_date'].fillna('N/A') # Fill the missing values in checkout_date column with N/A
combined_df['return_date'] = combined_df['return_date'].fillna('Not Returned') # Fill the missing values in checkout_date column with N/A that confirms that the books is still borrowed until now
combined_df['genre'] = combined_df['genre'].fillna('Unknown') # Fill the missing values in genre c olumn with N/A
combined_df['publisher'] = combined_df['publisher'].fillna('Unknown') # Fill the missing valuesin publisher column with N/A
combined_df['publication_year'] = combined_df['publication_year'].fillna('Unknown') # Fill the missing values in publication_year column with N/A
combined_df['pages'] = combined_df['pages'].fillna(0) # Fill the missing values in pages column with 0 to make it possible to perform matematical calculations later without any problems

In [36]:
print(combined_df.isnull().sum()) # display the number of missing values after cleaning

member_id            0
first_name           0
last_name            0
grade                0
neighborhood         0
membership_status    0
join_date            0
checkout_id          0
book_id              0
checkout_date        0
return_date          0
genre                0
pages                0
publication_year     0
publisher            0
dtype: int64


### Handle Orphan IDs

In [37]:
# Remove Orphan IDs
initial_count = len(combined_df)
combined_df = combined_df[combined_df['member_id'].isin(registrated_ids)].copy()

removed_orphans = initial_count - len(combined_df)

print(f"The number of removed Orphan IDs: {removed_orphans}")
print(f"The number of rows of the final cleaned DataFrame: {len(combined_df)}")

The number of removed Orphan IDs: 5
The number of rows of the final cleaned DataFrame: 422


### Save The Cleaned Data

In [38]:
combined_df.to_csv('task2_cleaned_data.csv', index=False)
print("Final Cleaned Dataset Saved Successfully!")

Final Cleaned Dataset Saved Successfully!


### Load The Cleaned Dataset

In [39]:
clean_df = pd.read_csv('task2_cleaned_data.csv')
print("Dataset Loaded Successfully.")

Dataset Loaded Successfully.


In [40]:
# Take a copy of the cleaned dataset

cleaned_df = clean_df.copy()

In [41]:
print(cleaned_df.head())

   member_id first_name last_name  grade neighborhood membership_status  \
0       1001      Salma   Ibrahim    8.0        Maadi            Active   
1       1002      Fares     Saleh    9.0        Maadi            Active   
2       1002      Fares     Saleh    9.0        Maadi            Active   
3       1003     Bassel    Hegazy    6.0        Maadi            Active   
4       1003     Bassel    Hegazy    6.0        Maadi            Active   

    join_date  checkout_id book_id checkout_date return_date      genre  \
0  2023-04-05       9025.0   525.0    2024-10-24  2024-11-07  Adventure   
1     Unknown       9013.0   501.0    2024-02-16  2024-02-29  Adventure   
2     Unknown       9095.0   507.0    2024-06-25  2024-07-07    Science   
3  2025-04-23       9051.0   513.0    2025-07-22  2025-08-18    Science   
4  2025-04-23       9068.0   501.0    2025-11-14  2025-12-03  Adventure   

   pages publication_year    publisher  
0  297.0           2015.0  Oasis Books  
1  128.0        

## Task3: Data Fairness & Version Control

In [42]:
# comparing neighborhoods

members = cleaned_df[['member_id', 'neighborhood']].drop_duplicates()['neighborhood'].value_counts()
checkouts = cleaned_df['neighborhood'].value_counts()

# Forming a dataframe
fairness_df = pd.DataFrame({"Members": members, "Checkouts": checkouts}).reset_index()

# Rename the index to be the neighborhood
fairness_df.rename(columns={'index': 'neighborhood'}, inplace=True)
print("fairness DataFrame:\n")
print(fairness_df)

fairness DataFrame:

  neighborhood  Members  Checkouts
0        Maadi       22        117
1    Nasr City       20        106
2   Heliopolis       18         92
3      Zamalek       14         72
4       Shubra        6         35
